# ⚡ City Energy Consumption Analysis & Prediction System

**Objective:** Identify patterns in daily electricity usage across city zones and predict next-day demand.

| Feature | Description |
|---|---|
| Date | Day of reading |
| ZoneID | City zone identifier (Z1–Z5) |
| AvgTemperature | Average temperature (°C) |
| Humidity | Humidity (%) |
| SpecialEvent | 0 = normal day, 1 = festival/sports |
| EnergyConsumption | Total energy used (kWh) |


## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Matplotlib dark style
plt.style.use('dark_background')
pd.set_option('display.float_format', '{:,.1f}'.format)
print('✓ Libraries loaded')


## 2. Synthetic Dataset Generation

We simulate 365 days × 5 zones = **1,825 records** with realistic seasonal temperature curves,
humidity anti-correlation, and zone-specific consumption profiles.


In [ ]:
from energy_analysis import generate_dataset, clean_and_preprocess

df_raw = generate_dataset(seed=42)
print(f'Shape: {df_raw.shape}')
df_raw.head(10)


## 3. Data Cleaning & Preprocessing

In [ ]:
df = clean_and_preprocess(df_raw.copy())
print('\nData types:')
print(df.dtypes)
print('\nDescriptive stats:')
df[['AvgTemperature','Humidity','EnergyConsumption']].describe().round(2)


## 4. Exploratory Analysis

### 4.1 Monthly Average Consumption per Zone


In [ ]:
monthly = df.groupby(['Month','ZoneID'])['EnergyConsumption'].mean().unstack()
monthly.index = ['Jan','Feb','Mar','Apr','May','Jun',
                  'Jul','Aug','Sep','Oct','Nov','Dec']
monthly.round(0)


### 4.2 Correlation between Features

In [ ]:
corr = df[['AvgTemperature','Humidity','SpecialEvent','EnergyConsumption']].corr()
corr.round(3)


### 4.3 Event vs Non-Event Consumption

In [ ]:
ev = df.groupby(['ZoneID','SpecialEvent'])['EnergyConsumption'].mean().unstack()
ev.columns = ['No Event (kWh)','Special Event (kWh)']
ev['Uplift %'] = ((ev['Special Event (kWh)'] / ev['No Event (kWh)']) - 1).mul(100).round(1)
ev.round(1)


## 5. Visualizations

### 5.1 Monthly Energy Usage Trends (Line Chart)

In [ ]:
from energy_analysis import plot_line_monthly
plot_line_monthly(df, 'outputs/plot_monthly_trends.png')
from IPython.display import Image
Image('outputs/plot_monthly_trends.png')


### 5.2 Feature Correlation Heatmap

In [ ]:
from energy_analysis import plot_heatmap_corr
plot_heatmap_corr(df, 'outputs/plot_correlation_heatmap.png')
Image('outputs/plot_correlation_heatmap.png')


### 5.3 Event vs Non-Event Bar Chart

In [ ]:
from energy_analysis import plot_bar_event
plot_bar_event(df, 'outputs/plot_event_comparison.png')
Image('outputs/plot_event_comparison.png')


### 5.4 Zone × Month Heatmap (Bonus)

In [ ]:
from energy_analysis import plot_daily_heatmap
plot_daily_heatmap(df, 'outputs/plot_zone_month_heatmap.png')
Image('outputs/plot_zone_month_heatmap.png')


## 6. Prediction Model

We compare **Random Forest** (primary) vs **Linear Regression** (baseline).

**Feature set:** AvgTemperature, Humidity, SpecialEvent, IsWeekend, Month, DayOfWeek, ZoneEncoded


In [ ]:
from energy_analysis import train_model, FEATURE_COLS

model, le, metrics = train_model(df)
print(f'\nRandom Forest R² : {metrics["rf_r2"]:.4f}')
print(f'Linear Regress R²: {metrics["lr_r2"]:.4f}')


### 6.1 Actual vs Predicted

In [ ]:
from energy_analysis import plot_model_performance
plot_model_performance(metrics, 'outputs/plot_model_performance.png')
Image('outputs/plot_model_performance.png')


### 6.2 Feature Importances

In [ ]:
import pandas as pd
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS)
importances.sort_values().plot.barh(figsize=(8,4), color='#3B82F6',
    title='Random Forest Feature Importances')
plt.tight_layout(); plt.show()


## 7. Quick Prediction Function

In [ ]:
def predict_zone(zone_id, temperature, humidity, special_event,
                 model=model, le=le, df=df):
    """Predict next-day energy for a given zone and weather inputs."""
    import datetime
    tomorrow = datetime.date.today() + datetime.timedelta(days=1)
    dow      = tomorrow.weekday()
    month    = tomorrow.month
    weekend  = int(dow >= 5)
    zenc     = le.transform([zone_id])[0]
    X = [[temperature, humidity, special_event, weekend, month, dow, zenc]]
    pred = model.predict(X)[0]
    zname = df.loc[df['ZoneID']==zone_id,'ZoneName'].iloc[0]
    print(f'Zone {zone_id} ({zname}) | {tomorrow} | ⚡ {pred:,.1f} kWh predicted')
    return pred

# Example predictions
predict_zone('Z1', temperature=28, humidity=55, special_event=1)
predict_zone('Z2', temperature=5,  humidity=80, special_event=0)
predict_zone('Z3', temperature=20, humidity=65, special_event=0)


## 8. Key Insights

| # | Insight |
|---|---|
| 1 | **Industrial zone (Z2)** consumes ≈2.5× more than Residential (Z3) |
| 2 | **Winter months (Dec–Jan)** show highest consumption due to heating needs |
| 3 | **Special events** boost University (Z5) demand by ~30%, the highest uplift |
| 4 | **Temperature correlation is weak globally** but strong within individual zones |
| 5 | **Random Forest (R²≈0.99, MAE≈105 kWh)** vastly outperforms Linear Regression |
| 6 | **ZoneEncoded** is the most important feature — zones have distinct baseline loads |
